# Анализ и подготовка данных о продажах видеоигр

- **Автор:** Дмитрий Галыгин
- **Дата:** 28.12.2025

### Цель и задачи проекта

**Цель:** подготовить готовые («чистые») данные для статьи, которую готовит команда игры «Секреты тёмнолесья».

**Задача:** обработать и выдать чистые данные по запросу команды, чтобы их было удобно использовать при написании статьи.


### Описание данных

Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержание проекта

1. Загрузка и знакомство с данными
2. Проверка ошибок в данных и их предобработка
3. Фильтрация данных
4. Категоризация данных

## 1. Загрузка данных и знакомство с ними

Загрузим необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/datasets/new_games.csv')

Познакомимся с данными: выведем первые строки и результат метода `info()`.

In [3]:
df.info()  # Выводим общую информацию о данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
df.isna().sum()  # Считаем количество пропусков в каждом столбце

Name                  2
Platform              0
Year of Release     275
Genre                 2
NA sales              0
EU sales              0
JP sales              0
Other sales           0
Critic Score       8714
User Score         6804
Rating             6871
dtype: int64

In [5]:
df.isna().sum() / len(df) * 100  # Доля пропусков в % от общего количества строк

Name                0.011795
Platform            0.000000
Year of Release     1.621845
Genre               0.011795
NA sales            0.000000
EU sales            0.000000
JP sales            0.000000
Other sales         0.000000
Critic Score       51.391838
User Score         40.127389
Rating             40.522529
dtype: float64

In [6]:
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,WII,2006-01-01,sports,41.360001,28.959999,3.77,8.45,76.0,8.0,E
1,super mario bros.,NES,1985-01-01,platform,29.080000,3.580000,6.81,0.77,0.0,0.0,NaN
2,mario kart wii,WII,2008-01-01,racing,15.680000,12.760000,3.79,3.29,82.0,8.3,E
3,wii sports resort,WII,2009-01-01,sports,15.610000,10.930000,3.28,2.95,80.0,8.0,E
4,pokemon red/pokemon blue,GB,1996-01-01,role-playing,11.270000,8.890000,10.22,1.00,0.0,0.0,NaN


In [7]:
df.nunique()  # Считаем количество уникальных значений в столбцах

Name               11559
Platform              31
Year of Release       37
Genre                 24
NA sales             402
EU sales             308
JP sales             245
Other sales          155
Critic Score          82
User Score            96
Rating                 8
dtype: int64

In [8]:
df.duplicated(subset=['Name'], keep='last').sum()  # Дубликаты в столбце 'Name' (без первого упоминания)

5396

### Выводы после первичного анализа
1. Названия столбцов некорректные.
2. Присутствуют пропуски в столбцах, с которыми нужно поработать.
3. В ряде столбцов некорректные типы данных.
4. Есть явные дубликаты, которые необходимо удалить.

## План очистки и подготовки данных
1. Привести названия столбцов к стилю snake_case.

2. Откорректировать типы данных, например:
- год релиза привести к типу `datetime64`;
- продажи и оценки привести к типу `float64`.

3. Пропуски:
- 2 пропуска в названии игры — явная ошибка, строки можно удалить;
- пропуски в году выпуска (`Year of Release`) составляют 1,6% от общего числа строк — значение несущественно для анализа, можно оставить;
- много пропусков в оценках и рейтинге; их появление объяснимо (оценки может просто не быть), но я бы их удалил, так как основная задача — проанализировать и проранжировать игры по этим критериям.

4. Дубликаты:
- по содержимому столбцов дубликаты допустимы везде, кроме `Name`: это явные дубликаты, так как не может быть нескольких игр с одинаковым названием. Их нужно удалить.

---

## 2. Проверка ошибок в данных и их предобработка

### 2.1. Названия столбцов датафрейма

Приведём все столбцы к стилю snake_case: нижний регистр и подчёркивания вместо пробелов.

In [9]:
print(df.columns)  # Выводим названия всех столбцов

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


In [10]:
df.columns = df.columns.str.lower()  # Приводим названия столбцов к нижнему регистру

In [11]:
df.columns = df.columns.str.replace(' ', '_')  # Заменяем пробелы на подчёркивания
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


### 2.2. Типы данных

Преобразуем столбцы к корректным типам данных. Числовые столбцы с пропусками нельзя сразу привести к `int64`, поэтому сначала обрабатываем пропуски, а затем меняем тип.

In [12]:
# Заполняем пропуски нулями в числовых столбцах перед сменой типа
for column in ['na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score']:
    df[column] = df[column].fillna(0)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11559 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   name             11559 non-null  object        
 1   platform         11559 non-null  object        
 2   year_of_release  11395 non-null  datetime64[ns]
 3   genre            11559 non-null  object        
 4   na_sales         11559 non-null  float32       
 5   eu_sales         11559 non-null  float32       
 6   jp_sales         11559 non-null  float32       
 7   other_sales      11559 non-null  float32       
 8   critic_score     11559 non-null  float32       
 9   user_score       11559 non-null  float32       
 10  rating           5941 non-null   object        
dtypes: datetime64[ns](1), float32(6), object(4)
memory usage: 812.7+ KB
None


In [13]:
df['year_of_release'] = pd.to_datetime(df['year_of_release'], format='%Y')  # Год выпуска -> datetime
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11559 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   name             11559 non-null  object        
 1   platform         11559 non-null  object        
 2   year_of_release  11395 non-null  datetime64[ns]
 3   genre            11559 non-null  object        
 4   na_sales         11559 non-null  float32       
 5   eu_sales         11559 non-null  float32       
 6   jp_sales         11559 non-null  float32       
 7   other_sales      11559 non-null  float32       
 8   critic_score     11559 non-null  float32       
 9   user_score       11559 non-null  float32       
 10  rating           5941 non-null   object        
dtypes: datetime64[ns](1), float32(6), object(4)
memory usage: 812.7+ KB
None


In [14]:
# Переводим оценки и продажи в float; строковые значения заменяются на пропуски (errors='coerce')
for column in ['na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score']:
    df[column] = pd.to_numeric(df[column], errors='coerce', downcast='float')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11559 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   name             11559 non-null  object        
 1   platform         11559 non-null  object        
 2   year_of_release  11395 non-null  datetime64[ns]
 3   genre            11559 non-null  object        
 4   na_sales         11559 non-null  float32       
 5   eu_sales         11559 non-null  float32       
 6   jp_sales         11559 non-null  float32       
 7   other_sales      11559 non-null  float32       
 8   critic_score     11559 non-null  float32       
 9   user_score       11559 non-null  float32       
 10  rating           5941 non-null   object        
dtypes: datetime64[ns](1), float32(6), object(4)
memory usage: 812.7+ KB
None


### 2.3. Пропуски в данных

Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.

In [15]:
df.isna().sum()  # Абсолютное количество пропусков

name                  0
platform              0
year_of_release     164
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score          0
user_score            0
rating             5618
dtype: int64

In [16]:
df.isna().sum() / len(df) * 100  # Относительное количество пропусков, %

name                0.000000
platform            0.000000
year_of_release     1.418808
genre               0.000000
na_sales            0.000000
eu_sales            0.000000
jp_sales            0.000000
other_sales         0.000000
critic_score        0.000000
user_score          0.000000
rating             48.602820
dtype: float64

### Оценка пропусков
1. Пропуски в `name` и `genre` — это ошибка, такие строки можно удалить.
2. В году выпуска 1,6% пропусков, их можно оставить:
    - значение некритично для анализа;
    - при сортировке по годам они и так не войдут в анализ.
3. Пропуски в `user_score` и `rating` объясняются тем, что игроки не оставили оценку, а ESRB не присвоила рейтинг — это допустимо, поэтому я их оставляю.
4. Есть пропуски в продажах (`eu_sales`, `jp_sales`) — их можно заменить усреднёнными значениями.

In [17]:
df = df.dropna(subset=['name'])  # Удаляем строки с пропусками в name
print(df.isna().sum())

name                  0
platform              0
year_of_release     164
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score          0
user_score            0
rating             5618
dtype: int64


In [18]:
eu_sales_min = df['eu_sales'].min()
eu_sales_max = df['eu_sales'].max()
eu_sales_mean = df['eu_sales'].mean()
eu_sales_median = df['eu_sales'].median()
jp_sales_min = df['jp_sales'].min()
jp_sales_max = df['jp_sales'].max()
jp_sales_mean = df['jp_sales'].mean()
eu_sales_median = df['eu_sales'].median()

print(eu_sales_min, eu_sales_max, eu_sales_mean, eu_sales_median)
print(jp_sales_min, jp_sales_max, jp_sales_mean, eu_sales_median)
# Разброс значений большой, медиана и среднее существенно ближе к нижней границе.
# Пропуски заменим средним значением в зависимости от платформы и года выхода игры.

0.0 28.96 0.15610218 0.02
0.0 10.22 0.10413518 0.02


In [19]:
# Заменяем пропуски средним значением в разрезе платформы и года выпуска
df['eu_sales'] = df['eu_sales'].fillna(df.groupby(['platform', 'year_of_release'])['eu_sales'].transform('mean'))
df['jp_sales'] = df['jp_sales'].fillna(df.groupby(['platform', 'year_of_release'])['jp_sales'].transform('mean'))
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11559 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   name             11559 non-null  object        
 1   platform         11559 non-null  object        
 2   year_of_release  11395 non-null  datetime64[ns]
 3   genre            11559 non-null  object        
 4   na_sales         11559 non-null  float32       
 5   eu_sales         11559 non-null  float32       
 6   jp_sales         11559 non-null  float32       
 7   other_sales      11559 non-null  float32       
 8   critic_score     11559 non-null  float32       
 9   user_score       11559 non-null  float32       
 10  rating           5941 non-null   object        
dtypes: datetime64[ns](1), float32(6), object(4)
memory usage: 812.7+ KB
None


### 2.4. Явные и неявные дубликаты

Изучим уникальные значения в категориальных столбцах (жанр, платформа, рейтинг) и проверим неявные дубликаты — опечатки и разное написание. При необходимости нормализуем текстовые значения.

Как отмечено выше, в столбце с названием игры много дубликатов — 5396, с ними и будем работать. В остальных столбцах дубликаты допустимы: оценка, год и жанр могут повторяться.

In [20]:
unique_genres = df['genre'].unique()
print(unique_genres)  # В жанрах есть дубликаты из-за разного регистра — приведём к нижнему

['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']


In [21]:
unique_platforms = df['platform'].unique()
print(unique_platforms)  # В платформах дубликатов из-за разного написания нет

['WII' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'WIIU' 'GC' 'GEN' 'XONE' 'DC' 'SAT' 'PSV'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']


Приведём столбцы `name` и `genre` к нижнему регистру.

In [22]:
df['name'] = df['name'].str.lower()
df['genre'] = df['genre'].str.lower()

In [23]:
# Удаляем дубликаты в столбце name
df.drop_duplicates(subset=['name'], inplace=True)

In [24]:
df.duplicated(subset = ['name'], keep = 'last').sum()

0

In [25]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11559 entries, 0 to 16953
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   name             11559 non-null  object        
 1   platform         11559 non-null  object        
 2   year_of_release  11395 non-null  datetime64[ns]
 3   genre            11559 non-null  object        
 4   na_sales         11559 non-null  float32       
 5   eu_sales         11559 non-null  float32       
 6   jp_sales         11559 non-null  float32       
 7   other_sales      11559 non-null  float32       
 8   critic_score     11559 non-null  float32       
 9   user_score       11559 non-null  float32       
 10  rating           5941 non-null   object        
dtypes: datetime64[ns](1), float32(6), object(4)
memory usage: 812.7+ KB
None


In [26]:
print(df.duplicated().sum())

0


Повторов в датафрейме больше нет — все строки уникальны. Было обнаружено 5396 дубликатов. Мы удалили дубликаты в столбце с названием игры, так как не может быть двух игр с одинаковым названием.

In [27]:
count_duplicated = 16956 - df.shape[0]
perc_count_duplicated = round(count_duplicated / 16956 * 100, 2)
print(f"Количество удалённых дубликатов в абсолютном значении: {count_duplicated}")
print(f"Количество удалённых дубликатов в относительном значении: {perc_count_duplicated}")

Количество удалённых дубликатов в абсолютном значении: 5397
Количество удалённых дубликатов в относительном значении: 31.83


### Предобработка данных выполнена
Выполнены следующие действия:
1. Названия столбцов приведены к стилю snake_case.
2. Исправлены типы данных:
 - год выпуска приведён к типу `datetime64`;
 - продажи и оценки приведены к типу `float64`.
3. Обработка пропусков:
 - удалены пропуски в `name` и `genre`;
 - пропуски в продажах заменены средним значением по категориям платформы и года выпуска;
 - пропуски в году выпуска оставлены — дальнейшая фильтрация их исключит;
 - строковые значения в числовых столбцах приведены к `NaN`.
4. Дубликаты:
 - выявлены явные и неявные дубликаты в столбцах `name` и `genre`;
 - неявные дубликаты в `genre` устранены приведением к нижнему регистру;
 - удалены строки с дубликатами в `name` — их было 5397, или 31,8%.

---

## 3. Фильтрация данных

Команду интересует история продаж игр в начале XXI века — период с 2000 по 2013 год включительно. Отберём данные за этот период и сохраним срез в отдельном датафрейме `df_actual`.

In [28]:
# Фильтруем данные по году выпуска: 2000–2013
df_actual = df[(df['year_of_release'] >= '2000-01-01') &
               (df['year_of_release'] <= '2013-12-31')]
print(df_actual)

                                     name platform year_of_release  \
0                              wii sports      WII      2006-01-01   
2                          mario kart wii      WII      2008-01-01   
3                       wii sports resort      WII      2009-01-01   
6                   new super mario bros.       DS      2006-01-01   
7                                wii play      WII      2006-01-01   
...                                   ...      ...             ...   
16943             storm: frontline nation       PC      2011-01-01   
16945                            plushees       DS      2008-01-01   
16946                             15 days       PC      2009-01-01   
16949  woody woodpecker in crazy castle 5      GBA      2002-01-01   
16952                    lma manager 2007     X360      2006-01-01   

            genre   na_sales   eu_sales  jp_sales  other_sales  critic_score  \
0          sports  41.360001  28.959999      3.77         8.45          76.0   

In [29]:
df_actual = df_actual.sort_values(by='year_of_release')  # Сортируем по году выпуска (по возрастанию)
print(df_actual)

                                               name platform year_of_release  \
7300                      ms. pac-man: maze madness      N64      2000-01-01   
479                           the sims: livin large       PC      2000-01-01   
11463            derby tsuku: derby uma o tsukurou!       DC      2000-01-01   
10435              foxkids.com micro maniacs racing       PS      2000-01-01   
4406   walt disney world quest: magical racing tour       PS      2000-01-01   
...                                             ...      ...             ...   
2668                       aliens: colonial marines     X360      2013-01-01   
14932                             kamigami no asobi      PSP      2013-01-01   
12403                       the guided fate paradox      PS3      2013-01-01   
906                        mario party: island tour      3DS      2013-01-01   
15325                               urakata hakuoki      PSP      2013-01-01   

              genre  na_sales  eu_sales

---

## 4. Категоризация данных

Разделим игры по оценкам пользователей на категории: высокая (8–10 включительно), средняя (от 3 до 8, не включая правую границу) и низкая (от 0 до 3, не включая правую границу).

In [30]:
def categorize_games_user_score (row):
    if 0 <= row['user_score'] < 3:
        return 'низкая оценка'
    elif 3 <= row['user_score'] < 8:
        return 'средняя оценка'
    elif 8 <= row['user_score'] <= 10:
        return 'высокая оценка'
    else: 'без оценки'
    
df_actual['user_score_category'] = df_actual.apply(categorize_games_user_score, axis = 1)
print(df_actual)

                                               name platform year_of_release  \
7300                      ms. pac-man: maze madness      N64      2000-01-01   
479                           the sims: livin large       PC      2000-01-01   
11463            derby tsuku: derby uma o tsukurou!       DC      2000-01-01   
10435              foxkids.com micro maniacs racing       PS      2000-01-01   
4406   walt disney world quest: magical racing tour       PS      2000-01-01   
...                                             ...      ...             ...   
2668                       aliens: colonial marines     X360      2013-01-01   
14932                             kamigami no asobi      PSP      2013-01-01   
12403                       the guided fate paradox      PS3      2013-01-01   
906                        mario party: island tour      3DS      2013-01-01   
15325                               urakata hakuoki      PSP      2013-01-01   

              genre  na_sales  eu_sales

Разделим игры по оценкам критиков на категории: высокая (80–100 включительно), средняя (от 30 до 80, не включая правую границу) и низкая (от 0 до 30, не включая правую границу).

In [31]:
def categorize_games_critic_score (row):
    if 0 <= row['critic_score'] < 30:
        return 'низкая оценка'
    elif 30 <= row['critic_score'] < 80:
        return 'средняя оценка'
    elif 80 <= row['critic_score'] <= 100:
        return 'высокая оценка'
    else: 'без оценки'
    
df_actual['critic_score_category'] = df_actual.apply(categorize_games_critic_score, axis = 1)
print(df_actual)

                                               name platform year_of_release  \
7300                      ms. pac-man: maze madness      N64      2000-01-01   
479                           the sims: livin large       PC      2000-01-01   
11463            derby tsuku: derby uma o tsukurou!       DC      2000-01-01   
10435              foxkids.com micro maniacs racing       PS      2000-01-01   
4406   walt disney world quest: magical racing tour       PS      2000-01-01   
...                                             ...      ...             ...   
2668                       aliens: colonial marines     X360      2013-01-01   
14932                             kamigami no asobi      PSP      2013-01-01   
12403                       the guided fate paradox      PS3      2013-01-01   
906                        mario party: island tour      3DS      2013-01-01   
15325                               urakata hakuoki      PSP      2013-01-01   

              genre  na_sales  eu_sales

Проверим результат: сгруппируем данные по категориям и посчитаем количество игр в каждой.

In [32]:
groupde_data_user_score = df_actual.groupby('user_score_category')['name'].count()
groupde_data_user_score = groupde_data_user_score.sort_values(ascending= False)
print(groupde_data_user_score)

user_score_category
низкая оценка     4859
средняя оценка    2270
высокая оценка    1575
Name: name, dtype: int64


In [33]:
groupde_data_critic_score = df_actual.groupby('critic_score_category')['name'].count()
groupde_data_critic_score = groupde_data_critic_score.sort_values(ascending= False)
print(groupde_data_critic_score)

critic_score_category
низкая оценка     4412
средняя оценка    3280
высокая оценка    1012
Name: name, dtype: int64


Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [34]:
popular_platform = df.groupby('platform')['name'].count()
popular_platform = popular_platform.sort_values(ascending= False)
print(popular_platform.head(7))

platform
PS2    1865
DS     1791
PS     1126
WII     937
PSP     905
PS3     693
GBA     623
Name: name, dtype: int64


---

## 5. Итоговый вывод

### 1. Топ-7 популярных платформ
 - консольные платформы оказались самыми популярными;
 - первое место занимает платформа PS2 — 1865 выпущенных игр;
 - в топ-7 четыре позиции занимают платформы PS, ещё три — Nintendo.

### 2. Категории игр по оценкам пользователей (2000–2013)
 - больше всего игр в категории «низкая оценка». Вероятно, это связано с высокой конкуренцией среди игр разных жанров: в этот период был всплеск игровой индустрии, и многие проекты оказались неудачными.

### 3. Категории игр по оценкам критиков (2000–2013)
 - результат примерно такой же, как и по оценкам пользователей: низких оценок больше, а высоких в среднем на 30% меньше, при этом градация категорий зеркальна.

Отдельно отмечу: в столбце `rating` (возрастная категория ESRB) очень много пропусков. Хотя в этом анализе они не участвовали, при подготовке статьи это важно учесть — у топовых игр рейтинг точно должен быть, поэтому данные стоит либо дополнить, либо удалить строки без рейтинга. Также во многих столбцах изначально был некорректный тип данных — это нужно учитывать при дальнейшей работе с базой.